```
  set Density (rho)                           = 1
  set Elastic modulus of the vessel wall (mu) = 1
  set Reference cross-sectional area (a0)     = 1
  set Reference pressure (p0)                 = 1
  set Reference radius (r0)                   = 1
  set Tube law exponent (m)                   = 0.5
  set Viscosity coefficient (eta_c)           = 1
```

In [ ]:
import sympy
from sympy import *
rho, mu, a0, p0, r0, m, eta_c, x, t = symbols('rho mu a_0 p_0 r_0 m eta_c x t')

A = Function('A')(x, t)
U = Function('U')(x, t)

dA = Function('\delta A')(x, t)
dU = Function('\delta U')(x, t)

fA = Function('f_A')(x, t)
fU = Function('f_U')(x, t)

# pressure
P = p0 + mu*( (A/a0)**m - 1 )

# fluxes
flux_A = A*U
flux_U = U**2/2 + P/rho

# manufactured RHS
fA_man = simplify(diff(A, t) + diff(flux_A, x) )
fU_man = simplify(diff(U, t) + diff(flux_U, x) + eta_c * U)

In [ ]:
# implicit function for ARKODE
LA = fA - diff(flux_A, x)
LU = fU - diff(flux_U, x) - eta_c * U

L = Matrix([LA, LU])
W = Matrix([A, U])
dW = Matrix([dA, dU])

# Frechet derivative of L in the direction dW (symbolic expression of J*dW)
eps = symbols('epsilon')
JdW = (L.subs({A: A + eps*dA, U: U + eps * dU}) - L).diff(eps).subs({eps: 0})

In [ ]:
JdW

In [ ]:
Aexp = a0*(1+.5*sin(2*pi*x-t*pi))
Aexp = x+1
Aexp = 0*x+1
Uexp = 0*x+1

dAexp = 0*x+1
dUexp = 0*x+1

A0exp = Aexp.subs({t:0})
U0exp = Uexp.subs({t:0})

s = {A: Aexp, U: Uexp,
     dA: dAexp, dU: dUexp}

fA_man_exp = fA_man.subs(s).simplify()
fU_man_exp = fU_man.subs(s).simplify()

Lexp = L.subs(s).subs({fA:fA_man_exp, fU:fU_man_exp})

JdWexp = JdW.subs(s)

In [ ]:
s = f"""
   set Exact solution = {Aexp}; {Uexp}
   set Initial condition = {A0exp}; {U0exp}
   set RHS expression = {fA_man_exp}; {fU_man_exp}

   set L(W) = {Lexp[0].simplify()}; {Lexp[1].simplify()}
   set dW = {dAexp}; {dUexp}
   set J*dW = {JdWexp[0,0].simplify()}; {JdWexp[1,0].simplify()}
"""

print(s.replace('pi', 'PI').replace('**', '^'))

In [ ]:
# HLL residual flux translated from blood_flow_system.h
A_L_sym, U_L_sym, A_R_sym, U_R_sym, bn_L, bn_R = symbols('A_L U_L A_R U_R bn_L bn_R')

def pressure(area):
    return p0 + mu*((area/a0)**m - 1)

def wave_speed(area):
    dpdA = diff(pressure(area), area)
    return sqrt(area/rho * dpdA)

c_L = wave_speed(A_L_sym)
c_R = wave_speed(A_R_sym)
U_bar = (U_L_sym + U_R_sym)/2
c_bar = (c_L + c_R)/2
s_L = U_bar - c_bar
s_R = U_bar + c_bar

# physical fluxes projected on the normal direction (b · n = bn)
FAL = A_L_sym * U_L_sym * bn_L
FUL = (0.5 * U_L_sym**2 + pressure(A_L_sym)/rho) * bn_L

FAR = A_R_sym * U_R_sym * bn_R
FUR = (0.5 * U_R_sym**2 + pressure(A_R_sym)/rho) * bn_R

FHLL_A = Piecewise(
    (FAL, s_L >= 0),
    (FAR, s_R <= 0),
    ((s_R*FAL - s_L*FAR + s_R*s_L*(A_R_sym - A_L_sym))/(s_R - s_L), True)
)

FHLL_U = Piecewise(
    (FUL, s_L >= 0),
    (FUR, s_R <= 0),
    ((s_R*FUL - s_L*FUR + s_R*s_L*(U_R_sym - U_L_sym))/(s_R - s_L), True)
)

FHLL_A, FHLL_U


In [ ]:
# Frechet derivative of HLL flux in directions dW_L and dW_R
dA_L, dU_L, dA_R, dU_R = symbols('\delta~A_L \delta~U_L \delta~A_R \delta~U_R')
dA_L, dU_L, dA_R, dU_R = symbols('trial_A_L trial_U_L trial_A_R trial_U_R')
eps = symbols('epsilon')

subs_eps = {
    A_L_sym: A_L_sym + eps*dA_L,
    U_L_sym: U_L_sym + eps*dU_L,
    A_R_sym: A_R_sym + eps*dA_R,
    U_R_sym: U_R_sym + eps*dU_R,
}

FHLL_A_eps = FHLL_A.subs(subs_eps)
FHLL_U_eps = FHLL_U.subs(subs_eps)

dFHLL_A = diff(FHLL_A_eps, eps).subs(eps, 0)
dFHLL_U = diff(FHLL_U_eps, eps).subs(eps, 0)

dFHLL_A, dFHLL_U


In [ ]:
# C++ code helpers for dFHLL_A and dFHLL_U
from sympy import ccode, cse, numbered_symbols

def ccode_with_defs(expr, name):
    temps, (main,) = cse(expr, symbols=numbered_symbols('t_'))

    def qualify_std(line):
        line = line.replace('pow(', 'std::pow(')
        line = line.replace('sqrt(', 'std::sqrt(')
        return line

    lines = [f'// {name}']
    lines += [qualify_std(f'const double {sym} = {ccode(val)};') for sym, val in temps]
    lines += [qualify_std(f'const double {name} = {ccode(main)};')]
    return "\n".join(lines)

print(ccode_with_defs(dFHLL_A, 'dFHLL_A'))
print()
print(ccode_with_defs(dFHLL_U, 'dFHLL_U'))
